In [ ]:
import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch version: 2.11.0+cpu
GPU available: False


In [ ]:
%pip install -q datasets

In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "cfilt/iitb-english-hindi",
    split="train[:10000]"
)

print("Number of examples:", len(dataset))
print("Dataset columns:", dataset.column_names)
print(dataset[0])

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  190MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.7kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

Number of examples: 10000
Dataset columns: ['translation']
{'translation': {'en': 'Give your application an accessibility workout', 'hi': 'अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें'}}


In [6]:
pairs = []

for item in dataset:
    en = item["translation"]["en"].strip()
    hi = item["translation"]["hi"].strip()

    if en and hi:
        pairs.append((en, hi))

print("Valid sentence pairs:", len(pairs))
print("\nExample:")
print("English:", pairs[1][0])
print("Hindi:", pairs[1][1])

# Remove duplicate sentence pairs
pairs = list(dict.fromkeys(pairs))

# Keep sentences within reasonable length limits
MAX_LEN = 20

filtered_pairs = []

for en, hi in pairs:
    en_words = en.split()
    hi_words = hi.split()

    if 1 <= len(en_words) <= MAX_LEN and \
       1 <= len(hi_words) <= MAX_LEN:
        filtered_pairs.append((en, hi))

print("Pairs before filtering:", len(pairs))
print("Pairs after filtering:", len(filtered_pairs))

print("\nSample filtered pair:")
print("English:", filtered_pairs[0][0])
print("Hindi:", filtered_pairs[0][1])

from collections import Counter

# Count English and Hindi words separately
en_counter = Counter()
hi_counter = Counter()

for en, hi in filtered_pairs:
    en_counter.update(en.lower().split())
    hi_counter.update(hi.split())

print("Unique English words:", len(en_counter))
print("Unique Hindi words:", len(hi_counter))

print("\nMost common English words:")
print(en_counter.most_common(10))

print("\nMost common Hindi words:")
print(hi_counter.most_common(10))

Valid sentence pairs: 10000

Example:
English: Accerciser Accessibility Explorer
Hindi: एक्सेर्साइसर पहुंचनीयता अन्वेषक
Pairs before filtering: 3440
Pairs after filtering: 3344

Sample filtered pair:
English: Give your application an accessibility workout
Hindi: अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
Unique English words: 1926
Unique Hindi words: 2157

Most common English words:
[('the', 658), ('of', 476), ('_', 396), ('a', 362), ('move', 234), ('to', 226), ('~', 215), ('onto', 208), ('file', 129), ('project', 118)]

Most common Hindi words:
[('को', 425), ('का', 396), ('करें', 295), ('के', 249), ('में', 229), ('एक', 191), ('(_', 188), ('पर', 178), ('a', 156), ('~', 155)]


In [7]:
# Inspect examples containing unusual symbols
for en, hi in filtered_pairs:
    if "_" in en or "~" in en or "_(" in hi:
        print("EN:", en)
        print("HI:", hi)
        print("-" * 40)

        # Limit the output
        if False:
            break

EN: _ Monitor Events
HI: घटनाओं को मानिटर करें (_ M)
----------------------------------------
EN: C _ lear Selection
HI: चुनाव को हटाएं (C _)
----------------------------------------
EN: _ Accessible
HI: पहुंचनीय
----------------------------------------
EN: Acti _ on
HI: कार्रवाई (_ o)
----------------------------------------
EN: Ap _ plication
HI: अनुप्रयोग
----------------------------------------
EN: Col _ lection
HI: संग्रह
----------------------------------------
EN: Co _ mponent
HI: घटक
----------------------------------------
EN: Des _ ktop
HI: डेस्कटोप
----------------------------------------
EN: _ Document
HI: प्रलेख
----------------------------------------
EN: H _ ypertext
HI: हाइपरपाठ
----------------------------------------
EN: _ Image
HI: छवि (_ I)
----------------------------------------
EN: Lo _ gin Helper
HI: प्रवेश सहायक
----------------------------------------
EN: _ Selection
HI: चयन (_ S)
----------------------------------------
EN: St _ reamable Content
HI: प्रवाहित 

In [8]:
import re

def has_ui_artifact(text):
    # Detect underscores surrounded by spaces or attached
    # to fragments of words, such as "C _ lear" or "Acti _ on"
    return bool(re.search(r"\w?_\w?|\s_\s", text))

clean_pairs = []

for en, hi in filtered_pairs:
    if has_ui_artifact(en) or has_ui_artifact(hi):
        continue

    clean_pairs.append((en, hi))

print("Before cleaning:", len(filtered_pairs))
print("After cleaning:", len(clean_pairs))
print("Removed:", len(filtered_pairs) - len(clean_pairs))

print("\nExamples retained:")
for en, hi in clean_pairs[:5]:
    print("EN:", en)
    print("HI:", hi)
    print("-" * 30)

Before cleaning: 3344
After cleaning: 2898
Removed: 446

Examples retained:
EN: Give your application an accessibility workout
HI: अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
------------------------------
EN: Accerciser Accessibility Explorer
HI: एक्सेर्साइसर पहुंचनीयता अन्वेषक
------------------------------
EN: The default plugin layout for the bottom panel
HI: निचले पटल के लिए डिफोल्ट प्लग-इन खाका
------------------------------
EN: The default plugin layout for the top panel
HI: ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका
------------------------------
EN: A list of plugins that are disabled by default
HI: उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से निष्क्रिय किया गया है
------------------------------


In [9]:
from collections import Counter

en_counter = Counter()
hi_counter = Counter()

for en, hi in clean_pairs:
    en_counter.update(en.lower().split())
    hi_counter.update(hi.split())

print("Clean sentence pairs:", len(clean_pairs))
print("Unique English tokens:", len(en_counter))
print("Unique Hindi tokens:", len(hi_counter))

Clean sentence pairs: 2898
Unique English tokens: 1776
Unique Hindi tokens: 2000


In [10]:
from collections import Counter

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

def build_vocab(counter):
    vocab = {
        token: idx
        for idx, token in enumerate(SPECIAL_TOKENS)
    }

    for word, frequency in counter.most_common():
        if word not in vocab:
            vocab[word] = len(vocab)

    return vocab

# Separate vocabularies for each language
en_vocab = build_vocab(en_counter)
hi_vocab = build_vocab(hi_counter)

print("English vocabulary size:", len(en_vocab))
print("Hindi vocabulary size:", len(hi_vocab))

print("\nSample English IDs:")
print(list(en_vocab.items())[:12])

print("\nSample Hindi IDs:")
print(list(hi_vocab.items())[:12])

English vocabulary size: 1780
Hindi vocabulary size: 2004

Sample English IDs:
[('<PAD>', 0), ('<SOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('the', 4), ('of', 5), ('a', 6), ('move', 7), ('to', 8), ('~', 9), ('onto', 10), ('file', 11)]

Sample Hindi IDs:
[('<PAD>', 0), ('<SOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('को', 4), ('का', 5), ('के', 6), ('में', 7), ('करें', 8), ('एक', 9), ('पर', 10), ('a', 11)]


In [11]:
from sklearn.model_selection import train_test_split

train_pairs, temp_pairs = train_test_split(
    clean_pairs,
    test_size=0.20,
    random_state=42
)

val_pairs, test_pairs = train_test_split(
    temp_pairs,
    test_size=0.50,
    random_state=42
)

print("Training pairs:", len(train_pairs))
print("Validation pairs:", len(val_pairs))
print("Test pairs:", len(test_pairs))

Training pairs: 2318
Validation pairs: 290
Test pairs: 290


In [12]:
from collections import Counter

en_counter = Counter()
hi_counter = Counter()

for en, hi in train_pairs:
    en_counter.update(en.lower().split())
    hi_counter.update(hi.split())

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

def build_vocab(counter):
    vocab = {token: i for i, token in enumerate(SPECIAL_TOKENS)}

    for word, _ in counter.most_common():
        if word not in vocab:
            vocab[word] = len(vocab)

    return vocab

en_vocab = build_vocab(en_counter)
hi_vocab = build_vocab(hi_counter)

print("English vocabulary:", len(en_vocab))
print("Hindi vocabulary:", len(hi_vocab))

English vocabulary: 1599
Hindi vocabulary: 1787


In [14]:
PAD_ID = en_vocab["<PAD>"]
SOS_ID = en_vocab["<SOS>"]
EOS_ID = en_vocab["<EOS>"]
UNK_ID = en_vocab["<UNK>"]

print("PAD:", PAD_ID)
print("SOS:", SOS_ID)
print("EOS:", EOS_ID)
print("UNK:", UNK_ID)

PAD: 0
SOS: 1
EOS: 2
UNK: 3


In [15]:
def encode_sentence(sentence, vocab, lowercase=False):
    if lowercase:
        sentence = sentence.lower()

    token_ids = [
        vocab.get(word, UNK_ID)
        for word in sentence.split()
    ]

    return [SOS_ID] + token_ids + [EOS_ID]


# Encode one training example
en, hi = train_pairs[0]

en_ids = encode_sentence(en, en_vocab, lowercase=True)
hi_ids = encode_sentence(hi, hi_vocab)

print("English:", en)
print("English IDs:", en_ids)

print("Hindi:", hi)
print("Hindi IDs:", hi_ids)

English: No type attribute specified.
English IDs: [1, 30, 137, 783, 506, 2]
Hindi: कोई फिल्टर का नाम निर्धारित नहीं है
Hindi IDs: [1, 66, 193, 5, 33, 267, 25, 28, 2]


In [16]:
import torch
from torch.nn.utils.rnn import pad_sequence

# Encode four training examples
examples = train_pairs[:4]

english_sequences = [
    torch.tensor(
        encode_sentence(en, en_vocab, lowercase=True)
    )
    for en, hi in examples
]

hindi_sequences = [
    torch.tensor(
        encode_sentence(hi, hi_vocab)
    )
    for en, hi in examples
]

# Pad each language separately
english_batch = pad_sequence(
    english_sequences,
    batch_first=True,
    padding_value=PAD_ID
)

hindi_batch = pad_sequence(
    hindi_sequences,
    batch_first=True,
    padding_value=PAD_ID
)

print("English batch shape:", english_batch.shape)
print(english_batch)

print("\nHindi batch shape:", hindi_batch.shape)
print(hindi_batch)

English batch shape: torch.Size([4, 9])
tensor([[  1,  30, 137, 783, 506,   2,   0,   0,   0],
        [  1,   4,  98,  14,   4, 138, 139, 293,   2],
        [  1, 157, 507,   2,   0,   0,   0,   0,   0],
        [  1, 373, 114,   2,   0,   0,   0,   0,   0]])

Hindi batch shape: torch.Size([4, 9])
tensor([[  1,  66, 193,   5,  33, 267,  25,  28,   2],
        [  1, 229, 574, 427,   6,  29, 575,   2,   0],
        [  1, 194, 268,   7,   2,   0,   0,   0,   0],
        [  1,  77, 119,   2,   0,   0,   0,   0,   0]])


In [18]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(
        self,
        input_vocab_size,
        embedding_dim,
        hidden_dim,
        padding_idx
    ):
        super().__init__()

        # Convert token IDs into trainable vectors
        self.embedding = nn.Embedding(
            input_vocab_size,
            embedding_dim,
            padding_idx=padding_idx
        )

        # Process the sequence one token at a time
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

    def forward(self, source):
        # source shape: [batch_size, sequence_length]
        embedded = self.embedding(source)

        # embedded shape: [batch_size, sequence_length, embedding_dim]
        outputs, (hidden, cell) = self.lstm(embedded)

        return outputs, hidden, cell

In [19]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 256

encoder = Encoder(
    input_vocab_size=len(en_vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    padding_idx=PAD_ID
)

# Use the English batch you created earlier
outputs, hidden, cell = encoder(english_batch)

print("Input shape:", english_batch.shape)
print("Encoder outputs:", outputs.shape)
print("Final hidden state:", hidden.shape)
print("Final cell state:", cell.shape)

Input shape: torch.Size([4, 9])
Encoder outputs: torch.Size([4, 9, 256])
Final hidden state: torch.Size([1, 4, 256])
Final cell state: torch.Size([1, 4, 256])


In [20]:
class Decoder(nn.Module):
    def __init__(
        self,
        output_vocab_size,
        embedding_dim,
        hidden_dim,
        padding_idx
    ):
        super().__init__()

        # Convert Hindi token IDs into vectors
        self.embedding = nn.Embedding(
            output_vocab_size,
            embedding_dim,
            padding_idx=padding_idx
        )

        # Process Hindi tokens sequentially
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Convert LSTM outputs into vocabulary scores
        self.fc = nn.Linear(
            hidden_dim,
            output_vocab_size
        )

    def forward(self, target, hidden, cell):
        # target: [batch_size, target_length]
        embedded = self.embedding(target)

        # Carry the encoder's states into the decoder
        outputs, (hidden, cell) = self.lstm(
            embedded,
            (hidden, cell)
        )

        # Predict a score for every Hindi vocabulary token
        logits = self.fc(outputs)

        return logits, hidden, cell

In [21]:
decoder = Decoder(
    output_vocab_size=len(hi_vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    padding_idx=PAD_ID
)

# Feed the Hindi batch and encoder states to the decoder
logits, dec_hidden, dec_cell = decoder(
    hindi_batch,
    hidden,
    cell
)

print("Hindi input shape:", hindi_batch.shape)
print("Decoder logits shape:", logits.shape)
print("Decoder hidden state:", dec_hidden.shape)
print("Decoder cell state:", dec_cell.shape)

Hindi input shape: torch.Size([4, 9])
Decoder logits shape: torch.Size([4, 9, 1787])
Decoder hidden state: torch.Size([1, 4, 256])
Decoder cell state: torch.Size([1, 4, 256])


In [22]:
from torch.nn.utils.rnn import pack_padded_sequence

class Encoder(nn.Module):
    def __init__(
        self, input_vocab_size, embedding_dim,
        hidden_dim, padding_idx
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            input_vocab_size,
            embedding_dim,
            padding_idx=padding_idx
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

    def forward(self, source, lengths):
        embedded = self.embedding(source)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, cell) = self.lstm(packed)

        return hidden, cell

In [23]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, source_lengths, target_input):
        hidden, cell = self.encoder(
            source, source_lengths
        )

        logits, _, _ = self.decoder(
            target_input, hidden, cell
        )

        return logits

In [24]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        en, hi = self.pairs[idx]

        source = encode_sentence(
            en, en_vocab, lowercase=True
        )
        target = encode_sentence(hi, hi_vocab)

        return (
            torch.tensor(source, dtype=torch.long),
            torch.tensor(target, dtype=torch.long)
        )


def collate_batch(batch):
    sources, targets = zip(*batch)

    source_lengths = torch.tensor(
        [len(s) for s in sources],
        dtype=torch.long
    )

    sources = pad_sequence(
        sources,
        batch_first=True,
        padding_value=PAD_ID
    )

    targets = pad_sequence(
        targets,
        batch_first=True,
        padding_value=PAD_ID
    )

    return sources, source_lengths, targets


train_loader = DataLoader(
    TranslationDataset(train_pairs),
    batch_size=32,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    TranslationDataset(val_pairs),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_batch
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 73
Validation batches: 10


In [25]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

EMBEDDING_DIM = 128
HIDDEN_DIM = 256

encoder = Encoder(
    len(en_vocab),
    EMBEDDING_DIM,
    HIDDEN_DIM,
    PAD_ID
)

decoder = Decoder(
    len(hi_vocab),
    EMBEDDING_DIM,
    HIDDEN_DIM,
    PAD_ID
)

model = Seq2Seq(encoder, decoder).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

print("Device:", device)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters()
        if p.requires_grad)
)

Device: cpu
Trainable parameters: 1683195


In [26]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0.0
    total_tokens = 0

    for sources, source_lengths, targets in loader:
        sources = sources.to(device)
        source_lengths = source_lengths.to(device)
        targets = targets.to(device)

        # Decoder receives everything except the final target token
        target_input = targets[:, :-1]

        # Labels are shifted one position to the right
        target_labels = targets[:, 1:]

        optimizer.zero_grad()

        logits = model(
            sources,
            source_lengths,
            target_input
        )

        # logits: [batch, target_length, vocab_size]
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            target_labels.reshape(-1)
        )

        loss.backward()

        # Prevent excessively large gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), 1.0
        )

        optimizer.step()

        valid_tokens = (
            target_labels != PAD_ID
        ).sum().item()

        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)


def evaluate_loss(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for sources, source_lengths, targets in loader:
            sources = sources.to(device)
            source_lengths = source_lengths.to(device)
            targets = targets.to(device)

            target_input = targets[:, :-1]
            target_labels = targets[:, 1:]

            logits = model(
                sources,
                source_lengths,
                target_input
            )

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                target_labels.reshape(-1)
            )

            valid_tokens = (
                target_labels != PAD_ID
            ).sum().item()

            total_loss += loss.item() * valid_tokens
            total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)

In [27]:
EPOCHS = 5

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion
    )

    val_loss = evaluate_loss(
        model, val_loader, criterion
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch 1/5 | Train Loss: 5.6305 | Val Loss: 4.8940
Epoch 2/5 | Train Loss: 4.2301 | Val Loss: 4.2419
Epoch 3/5 | Train Loss: 3.5552 | Val Loss: 3.8837
Epoch 4/5 | Train Loss: 3.0285 | Val Loss: 3.6622
Epoch 5/5 | Train Loss: 2.5771 | Val Loss: 3.4499


In [28]:
def translate(sentence, model, max_len=25):
    model.eval()

    source_ids = encode_sentence(
        sentence, en_vocab, lowercase=True
    )

    source = torch.tensor(
        [source_ids], dtype=torch.long, device=device
    )

    source_lengths = torch.tensor(
        [len(source_ids)], dtype=torch.long
    )

    with torch.no_grad():
        hidden, cell = model.encoder(
            source, source_lengths
        )

        # Begin generation with <SOS>
        current_token = torch.tensor(
            [[SOS_ID]], dtype=torch.long, device=device
        )

        generated_ids = []

        for _ in range(max_len):
            logits, hidden, cell = model.decoder(
                current_token, hidden, cell
            )

            # Select the highest-scoring next token
            next_id = logits[:, -1, :].argmax(dim=-1)
            token_id = next_id.item()

            if token_id == EOS_ID:
                break

            if token_id not in (PAD_ID, SOS_ID):
                generated_ids.append(token_id)

            current_token = next_id.unsqueeze(1)

    id_to_word = {
        idx: word for word, idx in hi_vocab.items()
    }

    return " ".join(
        id_to_word.get(idx, "<UNK>")
        for idx in generated_ids
    )

In [33]:
test_sentences = [
    "Give your application an accessibility workout",
    "The default plugin layout for the top panel",
    "A list of plugins that are disabled by default",
    "You are a dumb"
]

for sentence in test_sentences:
    print("English:", sentence)
    print("Hindi:", translate(sentence, model))
    print("-" * 50)

English: Give your application an accessibility workout
Hindi: अपने अनुप्रयोग को पहुंचनीयता का लाभ दें
--------------------------------------------------
English: The default plugin layout for the top panel
Hindi: ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका
--------------------------------------------------
English: A list of plugins that are disabled by default
Hindi: उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से निष्क्रिय किया गया है
--------------------------------------------------
English: You are a dumb
Hindi: एक फाइल को आप ढूंढ रहे हैं.
--------------------------------------------------


In [31]:
import pandas as pd

# Convert our cleaned sentence pairs into a table
df = pd.DataFrame(clean_pairs, columns=["English", "Hindi"])

# Display the first 20 sentence pairs
display(df.head(20))

,English,Hindi
0,Give your application an accessibility workout,अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
1,Accerciser Accessibility Explorer,एक्सेर्साइसर पहुंचनीयता अन्वेषक
2,The default plugin layout for the bottom panel,निचले पटल के लिए डिफोल्ट प्लग-इन खाका
3,The default plugin layout for the top panel,ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका
4,A list of plugins that are disabled by default,उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से नि...
5,Highlight duration,अवधि को हाइलाइट रकें
6,The duration of the highlight box when selecti...,पहुंचनीय आसंधि (नोड) को चुनते समय हाइलाइट बक्स...
7,Highlight border color,सीमांत (बोर्डर) के रंग को हाइलाइट करें
8,The color and opacity of the highlight border.,हाइलाइट किए गए सीमांत का रंग और अपारदर्शिता।
9,Highlight fill color,भराई के रंग को हाइलाइट करें


In [32]:
import copy

best_val_loss = evaluate_loss(
    model, val_loader, criterion
)

best_state = copy.deepcopy(model.state_dict())
patience = 3
epochs_without_improvement = 0
additional_epochs = 10

for epoch in range(additional_epochs):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion
    )

    val_loss = evaluate_loss(
        model, val_loader, criterion
    )

    print(
        f"Extra epoch {epoch + 1}/{additional_epochs} | "
        f"Train: {train_loss:.4f} | "
        f"Val: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Early stopping: validation loss stopped improving.")
        break

model.load_state_dict(best_state)

print("Best validation loss:", round(best_val_loss, 4))

Extra epoch 1/10 | Train: 2.1605 | Val: 3.3182
Extra epoch 2/10 | Train: 1.7887 | Val: 3.2572
Extra epoch 3/10 | Train: 1.4692 | Val: 3.1109
Extra epoch 4/10 | Train: 1.1862 | Val: 3.0978
Extra epoch 5/10 | Train: 0.9429 | Val: 3.0580
Extra epoch 6/10 | Train: 0.7372 | Val: 3.0414
Extra epoch 7/10 | Train: 0.5802 | Val: 3.0214
Extra epoch 8/10 | Train: 0.4476 | Val: 3.0390
Extra epoch 9/10 | Train: 0.3519 | Val: 3.0316
Extra epoch 10/10 | Train: 0.2869 | Val: 3.0270
Early stopping: validation loss stopped improving.
Best validation loss: 3.0214


In [34]:
print("Final training loss:",
      round(train_loss, 4))

print("Final validation loss:",
      round(val_loss, 4))

print("\nTest examples:")
for en, hi in test_pairs[:15]:
    print("English:  ", en)
    print("Expected: ", hi)
    print("Predicted: ", translate(en, model))
    print("-" * 60)

Final training loss: 0.2869
Final validation loss: 3.027

Test examples:
English:   Whether the window is maximized
Expected:  क्या विंडो सबसे बड़े आकार का होना चाहिए
Predicted:  विंडो की ऊंचाई का मान.
------------------------------------------------------------
English:   Ignore Directories:
Expected:  डायरेक्टरी का निर्माण करें
Predicted:  फाइल चयन करें
------------------------------------------------------------
English:   Subversion popup operations
Expected:  उर्ध्वाधर पैमाना
Predicted:  नमूना आपरेशन
------------------------------------------------------------
English:   LDTP
Expected:  एलडीटीपी
Predicted:  #-#-#-#-#gtk +. gtk-2-2. hi. po (gtk22 VERSION) #-#-#-#-#मोड (रूप) #-#-#-#-#libgnomeui. HEAD. hi. po (libgnomeui HEAD) #-#-#-#-#मोड
------------------------------------------------------------
English:   warning:
Expected:  चेतावनी
Predicted:  चेतावनीः
------------------------------------------------------------
English:   Calls
Expected:  #-#-#-#-#balsa. HEAD. hi. po (Balsa 2.

In [35]:
import re
import pandas as pd

# Keep the original data untouched
audit_df = pd.DataFrame(clean_pairs, columns=["English", "Hindi"])

# Patterns that may indicate translation-file artifacts
artifact_pattern = re.compile(
    r"#-#-#|\.po\b|gtk|gnome|libgnome|"
    r"\b(?:VERSION|HEAD)\b",
    flags=re.IGNORECASE
)

audit_df["possible_artifact"] = (
    audit_df["English"].str.contains(artifact_pattern, na=False)
    | audit_df["Hindi"].str.contains(artifact_pattern, na=False)
)

print("Total pairs:", len(audit_df))
print("Potentially suspicious pairs:", audit_df["possible_artifact"].sum())

display(
    audit_df[audit_df["possible_artifact"]][["English", "Hindi"]].head(30)
)

Total pairs: 2898
Potentially suspicious pairs: 76


,English,Hindi
35,Version,संस्करण
217,GNOME,गनोम
218,Default GNOME Theme,डिफ़ॉल्ट गनोम प्रसंग
440,GNOME Solitaire,गनोम सॉलिटेयर
464,GNOME Games web site,ग्नोम खेल वेबसाइट
870,Unrecognized desktop file Version '% s',अपरिचित डेस्कटाप फाइल संस्करण `% s '
1141,Western,#-#-#-#-#gtk +. gtk-2-2. hi. po (gtk22 VERSION...
1144,Turkish,#-#-#-#-#balsa. HEAD. hi. po (Balsa 2. x) #-#-...
1235,Class Elements,#-#-#-#-#gnome-desktop. HEAD. hi. po (gnome-de...
1287,Module,#-#-#-#-#gtk +. gtk-2-2. hi. po (gtk22 VERSION...


In [36]:
import re

artifact_pattern = re.compile(
    r"#-#-#|\.po\b|gtk|gnome|libgnome|"
    r"\b(?:VERSION|HEAD)\b",
    flags=re.IGNORECASE
)

def has_corrupted_translation(en, hi):
    # Metadata markers inside the Hindi target are strong evidence
    # of a corrupted translation field.
    return bool(artifact_pattern.search(hi))

audited_pairs = [
    (en, hi)
    for en, hi in clean_pairs
    if not has_corrupted_translation(en, hi)
]

print("Before cleaning:", len(clean_pairs))
print("After cleaning:", len(audited_pairs))
print("Removed:", len(clean_pairs) - len(audited_pairs))

Before cleaning: 2898
After cleaning: 2840
Removed: 58


In [37]:
remaining_suspicious = [
    (en, hi)
    for en, hi in audited_pairs
    if artifact_pattern.search(hi)
]

print("Remaining suspicious Hindi targets:", len(remaining_suspicious))
print("Sample cleaned pairs:")

for en, hi in audited_pairs[:10]:
    print(f"EN: {en}")
    print(f"HI: {hi}")
    print("-" * 40)

Remaining suspicious Hindi targets: 0
Sample cleaned pairs:
EN: Give your application an accessibility workout
HI: अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
----------------------------------------
EN: Accerciser Accessibility Explorer
HI: एक्सेर्साइसर पहुंचनीयता अन्वेषक
----------------------------------------
EN: The default plugin layout for the bottom panel
HI: निचले पटल के लिए डिफोल्ट प्लग-इन खाका
----------------------------------------
EN: The default plugin layout for the top panel
HI: ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका
----------------------------------------
EN: A list of plugins that are disabled by default
HI: उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से निष्क्रिय किया गया है
----------------------------------------
EN: Highlight duration
HI: अवधि को हाइलाइट रकें
----------------------------------------
EN: The duration of the highlight box when selecting accessible nodes
HI: पहुंचनीय आसंधि (नोड) को चुनते समय हाइलाइट बक्से की अवधि
------------------------------------

In [38]:
from sklearn.model_selection import train_test_split

train_pairs, temp_pairs = train_test_split(
    audited_pairs,
    test_size=0.20,
    random_state=42
)

val_pairs, test_pairs = train_test_split(
    temp_pairs,
    test_size=0.50,
    random_state=42
)

print("Training:", len(train_pairs))
print("Validation:", len(val_pairs))
print("Test:", len(test_pairs))

Training: 2272
Validation: 284
Test: 284


In [39]:
from collections import Counter

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

def build_vocab(counter):
    vocab = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}

    for word, _ in counter.most_common():
        if word not in vocab:
            vocab[word] = len(vocab)

    return vocab

en_counter = Counter()
hi_counter = Counter()

for en, hi in train_pairs:
    en_counter.update(en.lower().split())
    hi_counter.update(hi.split())

en_vocab = build_vocab(en_counter)
hi_vocab = build_vocab(hi_counter)

PAD_ID = en_vocab["<PAD>"]
SOS_ID = en_vocab["<SOS>"]
EOS_ID = en_vocab["<EOS>"]
UNK_ID = en_vocab["<UNK>"]

print("English vocabulary:", len(en_vocab))
print("Hindi vocabulary:", len(hi_vocab))

English vocabulary: 1573
Hindi vocabulary: 1691


In [40]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    TranslationDataset(train_pairs),
    batch_size=32,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    TranslationDataset(val_pairs),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_batch
)

test_loader = DataLoader(
    TranslationDataset(test_pairs),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_batch
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 71
Validation batches: 9
Test batches: 9


In [41]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 256

encoder = Encoder(
    len(en_vocab), EMBEDDING_DIM, HIDDEN_DIM, PAD_ID
)

decoder = Decoder(
    len(hi_vocab), EMBEDDING_DIM, HIDDEN_DIM, PAD_ID
)

model = Seq2Seq(encoder, decoder).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

print("Fresh model initialized!")

Fresh model initialized!


In [42]:
import copy

best_val_loss = float("inf")
best_state = copy.deepcopy(model.state_dict())

patience = 3
epochs_without_improvement = 0
max_epochs = 15

for epoch in range(max_epochs):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion
    )

    val_loss = evaluate_loss(
        model, val_loader, criterion
    )

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Early stopping triggered.")
        break

model.load_state_dict(best_state)

print("\nBest validation loss:", round(best_val_loss, 4))

Epoch 01 | Train Loss: 5.5387 | Val Loss: 4.9815
Epoch 02 | Train Loss: 4.2334 | Val Loss: 4.3808
Epoch 03 | Train Loss: 3.6107 | Val Loss: 4.0428
Epoch 04 | Train Loss: 3.1298 | Val Loss: 3.8401
Epoch 05 | Train Loss: 2.6917 | Val Loss: 3.6696
Epoch 06 | Train Loss: 2.2695 | Val Loss: 3.5210
Epoch 07 | Train Loss: 1.8974 | Val Loss: 3.3577
Epoch 08 | Train Loss: 1.5614 | Val Loss: 3.3158
Epoch 09 | Train Loss: 1.2665 | Val Loss: 3.2471
Epoch 10 | Train Loss: 1.0091 | Val Loss: 3.2102
Epoch 11 | Train Loss: 0.7961 | Val Loss: 3.2269
Epoch 12 | Train Loss: 0.6249 | Val Loss: 3.2035
Epoch 13 | Train Loss: 0.4893 | Val Loss: 3.1839
Epoch 14 | Train Loss: 0.3858 | Val Loss: 3.2154
Epoch 15 | Train Loss: 0.3128 | Val Loss: 3.2412

Best validation loss: 3.1839


In [43]:
for en, hi in test_pairs[:15]:
    print("English:  ", en)
    print("Expected: ", hi)
    print("Predicted:", translate(en, model))
    print("-" * 60)

English:   CVS: Log file / directory
Expected:  कार्य डायरेक्टरी
Predicted: कार्य डायरेक्टरी
------------------------------------------------------------
English:   Move ~ a onto the ten of clubs.
Expected:  ~ a को चिड़ी का दहला पर ले जाएँ
Predicted: ~ a को चिड़ी का दहला पर घुमाएँ.
------------------------------------------------------------
English:   Documents
Expected:  खाली दस्तावेज
Predicted: दस्तावेज़
------------------------------------------------------------
English:   None
Expected:  कुछ नहीं
Predicted: कोई नहीं
------------------------------------------------------------
English:   Sample Plugin for Anjuta.
Expected:  Anjuta के लिए नमूना प्लगिन.
Predicted: टैग लिस्ट प्लगइन
------------------------------------------------------------
English:   Stash Message (Optional)
Expected:  सत्र
Predicted: संदेश सहेजें
------------------------------------------------------------
English:   Move ~ a onto the eight of spades.
Expected:  ~ a को हुकुम का अट्ठा पर घुमाएँ.
Predicted: ~ a को ए